# Config File — Pipeline Configuration

Central configuration for the GTA Real Estate ML Pipeline. Every notebook imports this file with `%run config.ipynb` so all paths, column names, thresholds, and functions are defined in one place.

**To adapt to a new dataset:** update `RAW_PATH` only — everything else flows from it unless column names have changed.

**Sections:**
1. Libraries and file paths
2. Target variable and column reconciliation
3. Columns to drop
4. Rows to drop
5. Audit columns
6. Thresholds
7. Encoding
8. VIF columns
9. Final feature list
10. Required columns
11. Schema validation function
12. Pipeline health check function

## 1. Libraries and File Paths

Core imports and file path definitions. `RAW_PATH` is the only value that changes when a new dataset arrives. `PROCESSED_PATH` and `FEATURES_PATH` are the outputs from preprocessing and feature engineering respectively.

In [1]:
import pandas as pd
import numpy as np

In [2]:
RAW_PATH = "merged_dataset_v3.csv"
PROCESSED_PATH = 'cleaned_dataset.csv'
FEATURES_PATH = 'features_listing.csv'

## 2. Target Variable and Column Reconciliation

Defines the target variable and the mappings needed to standardise the raw dataset:

- `DIFFERENT_SOLD_PRICES_COLS` — the two sold price columns from the merge that get unified via `combine_first`
- `UPDATED_NAME_COLS` — renames raw columns to the clean standard names used throughout the pipeline
- `FILL_HOMEDATA_COLS` — maps scraped `homeData_*` columns to the clean columns they fill when the clean column is null
- `HOMEDATA_REMAINDER` — the homeData columns that were actually used (kept during the drop step, all others removed)

In [3]:
TARGET = 'sold_price'

DIFFERENT_SOLD_PRICES_COLS = ("sold_price_x", "sold_price_y")
 
UPDATED_NAME_COLS = {
    'square_feet' : 'square_feet_num', #Note: new dataset is string range e.g. '700-1100 sqft' need parse_sqft() before use 
    "lot_size"    : 'lot_size_sqft', #Note: new dataset is dimension string e.g '116.73 x 197.92 Feet' needs parsing, mmixed units (Feet/acres)
}

FILL_HOMEDATA_COLS = {
    "homeData_beds"                                                  : 'bedrooms',
    "homeData_baths"                                                : 'bathrooms',
    "homeData_addressInfo_city"                                     : 'city',
    "homeData_addressInfo_centroid_centroid_latitude"               : 'latitude',
    "homeData_addressInfo_centroid_centroid_longitude"              : 'longitude',
    "homeData_lotSize_amount"                                       : 'lot_size_sqft',

}

HOMEDATA_REMAINDER =list(FILL_HOMEDATA_COLS.keys())




### 2.1 listing_price Note

`listing_price` comes in as a string dtype in `reduced_dataset.csv`. Convert in preprocessing with:

```python
df['listing_price'] = df['listing_price'].apply(
    lambda x: float(x.replace('$', '').replace(',', '')) if isinstance(x, str) else x
).astype(float)
```

**TODO — listing_price imputation strategy pending (see group discussion)**  
Option 1: Cross-city ratio imputation using Mississauga / Richmond Hill sold-to-listing ratio  
Option 2: Drop affected rows — run diagnostic checks first before deciding

## 3. Columns to Drop

- `COL_TO_DROP` — identifier columns, date columns superseded by engineered features, and columns with ~100% missing values. Applied in bulk during preprocessing.
- `CATEGORICAL_COLS` — columns treated as categorical for imputation (mode) and encoding (OHE).

In [4]:
COL_TO_DROP = [
    "address",
    'mls_number',
    'source',
    'listing_date',
    'sale_date',
    'walk_score',
    'zolo_estimate',
    'missing_count',
    
]

CATEGORICAL_COLS = ['city', 'property_type']

## 4. Rows to Drop

`ROW_NULL_DROP` — columns where a null value makes a row unusable for modeling. Any row missing any of these is dropped via `dropna(subset=ROW_NULL_DROP)` in preprocessing.

In [5]:
ROW_NULL_DROP = [
    'sold_price',
    'listing_price',
    'bedrooms',
    'bathrooms',
    'city',
    'property_type',
]



## 5. Audit Columns

`CHECK_COL` — the columns checked for missingness after the homeData recovery step in preprocessing. Gives a focused view of the columns that matter for modeling rather than printing all 66 columns.

In [6]:
CHECK_COL = [
    'sold_price','listing_price','bedrooms','bathrooms',
    'city','latitude','longitude','property_type',
    'square_feet_num', 'lot_size_sqft', 'taxes', 'median_income',
    'population', 'parking_spaces', 'house_age'
]

## 6. Thresholds

All business logic limits in one place — change here and it applies everywhere:

- `MIN_LISTING_PRICE` — rows below this are data errors (e.g. $1 listing price)
- `FILL_BEDROOMS` — replace 0-bedroom listings with 1 (parking/storage units, not residential)
- `ROW_THRESH_SPARSE` — diagnostic threshold for flagging sparse rows. **Note:** tuned for the old dataset which had ~63% missingness. New dataset max is ~15% — revisit this threshold in EDA.
- `N_PROPERTY_TYPE_TOP` — number of property types to keep before bucketing the rest as `'Others'`

In [7]:
MIN_LISTING_PRICE = 1000
FILL_BEDROOMS = 1
ROW_THRESH_SPARSE = 24  # meant for old dataset, new dataset max missingness ~15% revisit this threshold in eda
N_PROPERTY_TYPE_TOP = 8

## 7. Encoding

- `NEW_AGE_ORDER` — ordinal mapping for `house_age`. Covers all variant spellings found across dataset versions (e.g. `'6-10'` and `'6-10 Years'` both map to 2). `'Unknown'` and `'No Data'` map to -1.
- `NORMLIZE_PROPERTY_TYPE` — normalizes known typo variants in `property_type` before one-hot encoding. `'Att/Row/Twnhouse'` and `'Att/Row/Townhouse'` are the same category.

In [8]:
NEW_AGE_ORDER = {"New": 0, 
                 "0-5 years": 1,
                 "0-5 years": 1,
                 '6-10': 2,
                 '6-10 Years': 2,  
                 "6 -15 Years":2,
                 "6-15":2,
                 "11-15": 3,
                 "11-15 Years": 3,
                 '16-30': 4,
                 '16-30 Years': 4,
                 '31-50': 5,
                 '31-50 Years': 5, 
                 '51-99': 6,
                 '51-99 Years': 6,
                 '100+ years': 7,
                 '100+ Years': 7,
                 'Unknown': -1,
                 'No Data': -1}

NORMLIZE_PROPERTY_TYPE = {
    'Att/Row/Twnhouse' : "Att/Row/Townhouse",
}

## 8. VIF Columns

`DROP_COLS_VIF` — columns removed after Variance Inflation Factor analysis due to high multicollinearity. These features were identified as redundant in the modeling stage and are dropped before building the final feature set.

In [9]:
DROP_COLS_VIF = [
    'bed_plus_baths',
    'bath_bed_ratio',
    "taxes_per_sqft",
    'latitude',
    'longitude',
    'listing_year',
    'square_feet_num',
    'median_income',
    'taxes',
    'bedrooms',
    'listing_month',
]

## 9. Final Feature List

- `FEATURES_FINAL` — the core numeric and encoded features passed to the model. OHE columns from `city` and `property_type` are appended dynamically in the preprocessing notebook.
- `ONE_HOT_COLS` — columns that get one-hot encoded via `pd.get_dummies` with `drop_first=True`.

In [10]:
FEATURES_FINAL = [
    'week_of_month',
    'listing_price',
    'bathrooms',
    'parking_spaces',
    'encoded_house_age',
]

ONE_HOT_COLS = ['property_type', 'city']

## 10. Required Columns

`NEEDED_COLS` — the minimum set of columns the pipeline expects to find after loading the raw dataset. Used by `checking_function` to validate the schema and warn early if any expected column is missing or renamed.

In [11]:
NEEDED_COLS = [
    'listing_price',
    'bedrooms',
    'bathrooms',
    'city',
    'property_type',
    'latitude',
    'longitude',
    'days_since_sold',
    'parking_spaces'
]

## 11. Schema Validation Function

`checking_function` — call at the top of every notebook right after loading the data.

- `pre_check=True` — use on the raw file before reconciliation. Verifies `sold_price_x` and `sold_price_y` both exist.
- `pre_check=False` — use after reconciliation. Verifies `sold_price` exists and has no nulls.

In both modes, checks that all `NEEDED_COLS` are present and prints a warning with available columns if any are missing.

In [12]:
def checking_function(df, pretext = '', pre_check = False):
    word = f'[{pretext}]' if pretext else ""

    if pre_check:
        for col in DIFFERENT_SOLD_PRICES_COLS:
            if col not in df.columns:
                raise ValueError(f"{word} Raw file is missing {col} - check it is merged dataset"
                )

    else:
        if TARGET not in df.columns:
            raise ValueError(f"{word} '{TARGET} column is missing,  was it the combine_first step run "
                )
        nanulls = df[TARGET].isnull().sum()
        if nanulls > 0:
             raise ValueError(f"{word} '{TARGET} has {nanulls} nulls,  was it the combine_first step run "
                )
    
    not_there = [i for i in NEEDED_COLS if i not in df.columns]
    if not_there:
        print(f"{word} expected columns not found: {not_there}")
        print(f" available columns: {df.columns.tolist()}")

    else:
        print(f"{word} Schema Ok shape {df.shape}")

    return True

## 12. Pipeline Health Check Function

`health_stat` — call at the bottom of each notebook before saving the output file. Prints a summary of what happened in that stage: rows and columns dropped, nulls remaining, and the output file path.

```python
# Usage
health_stat(raw_df, df_final, output_path=FEATURES_PATH, label='Preprocessing')
```

In [13]:
def health_stat(df_in, df_out, output_path =None, label = ''):

    dropped_rows = len(df_in) - len(df_out)
    dropped_col = df_in.shape[1] - df_out.shape[1]
    pct_null = df_out.isnull().sum().sum() / df_out.size * 100
    divider_variable = '=' * 55

    print(f"\n {divider_variable}")
    print(f" Health Check {label}")
    print(f" {divider_variable}")
    print(f" Input shape {df_in.shape}")
    print(f" Output shape {df_out.shape}")
    print(f" Rows dropped {dropped_rows:,}  ({dropped_rows / max(len(df_in), 1) * 100:.1f}%)")
    print(f" Cols dropped {dropped_col}")
    print(f" Nulls left  {pct_null:.2f}%")
    if output_path:
        print(f" Saved to   : {output_path}")
    print(f'{divider_variable}')